# SpherIDiff Variants of the Pilot

This notebook trains the smaller SpherIDiff variants of the pilot and a shorter run of the main configuration, and evaluates them under the fixed final protocol. The variants `spheridiff_xs` (0.27M parameters), `spheridiff_s` (0.35M) and `spheridiff_m` (0.60M) are trained for the same 45,000 steps as the main pilot. They are small, so the three runs share the first GPU as parallel processes. The second GPU trains `spheridiff_b4` for fewer steps (`pilot_spheridiff_b4_short`), chosen to match the training time of the largest WIND model in the main pilot. All runs use the band limit of the main pilot, so that the comparison with the main pilot stays controlled.

The notebook uses the ERA5 cache at 5.6 degrees, attached as a Kaggle input, and the repository code, cloned from GitHub. It is designed for a background run, in which Kaggle executes the whole notebook and keeps the files of `/kaggle/working` when it ends. The results are packed into `pilot_variants_results.zip` in the working folder.

After the training, every run is evaluated under the same fixed final protocol as the main pilot (`configs/protocol/final_v1.yaml`) on the held-out test year. When the output of the main pilot is attached as a Kaggle input, the runs are compared with the WIND runs and the main SpherIDiff run of the pilot.

In [ ]:
import os
import sys
import time
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
VARIANT_EXPERIMENTS = ["pilot_spheridiff_xs", "pilot_spheridiff_s", "pilot_spheridiff_m"]
TIME_MATCHED_EXPERIMENT = "pilot_spheridiff_b4_short"
RUN_NAMES = VARIANT_EXPERIMENTS + [TIME_MATCHED_EXPERIMENT]
REFERENCE_RUNS = ["pilot_wind_s", "pilot_wind_m", "pilot_wind_l", "pilot_spheridiff_b4"]
ARCHIVE = WORK_ROOT / "pilot_variants_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. The data cache is read from the attached dataset `thesyeddaniyal/era5-pilot-6degrees`, at `/kaggle/input/datasets/thesyeddaniyal/era5-pilot-6degrees`. The folder that holds `stats.npz` is used as the cache root. When that input path does not exist, the whole of `/kaggle/input` is searched for `stats.npz`.

The runs of the main pilot (`pilot_wind_s`, `pilot_wind_m`, `pilot_wind_l` and `pilot_spheridiff_b4`) are used as references when they are attached as a Kaggle input, for example as the output of the main pilot notebook. They are found by the folder name and the `evaluations.jsonl` file inside. Without them, the run is trained and stored, and the comparison is left out.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
DATASET_ROOT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"


def locate(marker, root):
    for current, _, files in os.walk(root):
        if marker in files:
            return Path(current)
    raise FileNotFoundError(f"{marker} was not found in {root}")


def locate_reference(name):
    for root in (INPUT_ROOT, WORK_ROOT):
        for current, _, files in os.walk(root):
            if Path(current).name == name and "evaluations.jsonl" in files:
                return Path(current).parent
    return None


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
CACHE_ROOT = locate("stats.npz", DATASET_ROOT if DATASET_ROOT.exists() else INPUT_ROOT)
REFERENCE_ROOT = locate_reference(REFERENCE_RUNS[0])
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("data:", CACHE_ROOT)
print("reference runs:", REFERENCE_ROOT)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

Each job is a separate process with its own log file in `runs/_logs`. The three variants run as three parallel processes on the first GPU, which is possible because their memory use is far below the memory of the card. The shorter run of the main configuration uses the second GPU. With one GPU, all four processes share it. The notebook prints every line that a job writes as soon as it appears, prefixed with the elapsed time, the GPU and the job name. The same lines are kept in `runs/_logs/parallel.log`.

The jobs and the packing of the results share one cell with a `try` and a `finally`. When a job fails, or the cell is interrupted, the figures, the status file and the archive are still produced. The archive is also refreshed every 30 minutes during the run. The jobs have no time limit of their own and run until Kaggle ends the session. Checkpoints at 25%, 50% and 75% of the steps are kept. The final evaluation starts when all training jobs have ended. It writes `final_evaluation.json` into each run folder, and the table `final_evaluation_summary.txt` lists all runs under the same protocol.

In [ ]:
from spheridiff.kaggle.finalise import finalise, refresh_archive
from spheridiff.kaggle.jobs import final_job, place_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
LOGS = RUNS_ROOT / "_logs"
TRAINING_CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
EVALUATION_CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, LOGS, CACHE_ROOT)
TRAINING_PLACEMENT = [(0, [train_job(name)]) for name in VARIANT_EXPERIMENTS] + [
    (1, [train_job(TIME_MATCHED_EXPERIMENT)])
]
EVALUATION_PLACEMENT = [(0, [final_job(name, FINAL_PROTOCOL)]) for name in VARIANT_EXPERIMENTS] + [
    (1, [final_job(TIME_MATCHED_EXPERIMENT, FINAL_PROTOCOL)])
]
COMPARISONS = [(run, reference) for run in RUN_NAMES for reference in REFERENCE_RUNS] if REFERENCE_ROOT else []


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
EVALUATOR = None
try:
    RUNNER = ParallelRunner(place_queues(TRAINING_PLACEMENT, GPUS), TRAINING_CONTEXT, refresh=refresh)
    RUNNER.run()
    EVALUATOR = ParallelRunner(place_queues(EVALUATION_PLACEMENT, GPUS), EVALUATION_CONTEXT, refresh=refresh)
    EVALUATOR.run()
finally:
    OUTCOMES = [*(RUNNER.outcomes() if RUNNER else []), *(EVALUATOR.outcomes() if EVALUATOR else [])]
    finalise(
        RUNS_ROOT,
        ARCHIVE,
        OUTCOMES,
        RUN_NAMES,
        comparisons=COMPARISONS,
        reference_root=REFERENCE_ROOT,
        at_final=True,
        final_names=RUN_NAMES,
        final_reference_names=REFERENCE_RUNS if REFERENCE_ROOT else [],
    )
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The summaries list the steps, training time, speed and final loss of every run. When the reference runs are available, the comparison summary lists, for each run against each reference, the step at which it reaches the final error of the reference and stays there, the errors at fixed fractions of training, the mean error over the whole curve, and the paired difference of the final errors with a 95% interval over the evaluation windows. The paired difference uses the last evaluation of each run, because the step counts differ. The figures show the runs together with the references, against steps, samples seen and training time. The final evaluation table lists, for each run, the number of parameters, the trained steps, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row.

In [ ]:
import json

for name in RUN_NAMES:
    summary = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

comparison = RUNS_ROOT / "comparison_summary.txt"
if comparison.exists():
    print(comparison.read_text(encoding="utf-8"))

final_summary = RUNS_ROOT / "final_evaluation_summary.txt"
if final_summary.exists():
    print(final_summary.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))